# Piper Vietnamese (sea-g2p) - Fine-tune on Kaggle (T4 x2 / P100)
Multi-speaker, medium quality (22050 Hz), warmstart from LibriTTS-R medium checkpoint.

In [ ]:
!nvidia-smi

In [ ]:
# Config for Kaggle
import os

REPO_URL    = "https://github.com/18200245/piper-seag2p.git"
DATASET_DIR = "/kaggle/input/datasets/xkorenguyenthinh/nhieugiong"
WORK        = "/kaggle/working/piper_work"
OUT         = "/kaggle/working/piper_out"
VOICE       = "nhieugiong"

BASE_REPO         = "rhasspy/piper-checkpoints"
BASE_CKPT_IN_REPO = "en/en_US/libritts_r/medium/best.ckpt"

# T4 has 16GB VRAM per GPU -> batch_size 12-16 with length_bucketing + duration filter (<=15s) fits safely
BATCH_SIZE   = 12
MAX_EPOCHS   = 300
MIN_DURATION = 0.5
MAX_DURATION = 15.0

os.makedirs(WORK, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

In [ ]:
!apt-get -qq update && apt-get -qq install -y espeak-ng build-essential > /dev/null
!rm -rf {WORK}/piper
!git clone {REPO_URL} {WORK}/piper
%cd {WORK}/piper
!pip install -q -e ".[train,vi]" huggingface_hub soundfile
!bash build_monotonic_align.sh

In [ ]:
# Sanity check: sea-g2p phonemizer
import sys
if f"{WORK}/piper/src" not in sys.path:
    sys.path.insert(0, f"{WORK}/piper/src")

from piper.phonemize_vietnamese import VietnamesePhonemizer
print(VietnamesePhonemizer().phonemize("Chương hai nghìn tám trăm mười. Bầy Gương."))

In [ ]:
# Locate CSV and audio dir in Kaggle Input, clean invalid lines & filter extreme durations (prevents T4 OOM)
import glob, collections, os
import soundfile as sf

csv_candidate = os.path.join(DATASET_DIR, "metadata.csv")
if os.path.exists(csv_candidate):
    raw_csv_path = csv_candidate
else:
    cands = sorted(glob.glob("/kaggle/input/**/metadata.csv", recursive=True) + glob.glob("/kaggle/input/**/*.csv", recursive=True))
    print("CSV candidates:", cands)
    assert cands, "No CSV found under /kaggle/input!"
    raw_csv_path = cands[0]

audio_dir = os.path.dirname(raw_csv_path)
print("Raw CSV:", raw_csv_path)
print("Audio dir:", audio_dir)

good, bad, missing, too_long = [], [], 0, 0
for i, line in enumerate(open(raw_csv_path, encoding="utf-8-sig"), 1):
    line = line.rstrip("\r\n")
    if not line.strip():
        continue
    r = line.split("|", 2)
    if i == 1 and r[0].strip().lower() in ("file", "filename", "path", "audio", "wav"):
        continue
    if len(r) != 3 or not all(x.strip() for x in r):
        bad.append((i, line))
        continue
    wav_rel, spk, txt = r[0].strip(), r[1].strip(), r[2].strip()
    wav_full = os.path.join(audio_dir, wav_rel)
    if not os.path.exists(wav_full):
        missing += 1
        continue
    try:
        dur = sf.info(wav_full).duration
    except Exception:
        bad.append((i, line))
        continue
    if not (MIN_DURATION <= dur <= MAX_DURATION):
        too_long += 1
        continue
    good.append((wav_rel, spk, txt))

csv_path = os.path.join(WORK, "metadata_clean.csv")
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("\n".join("|".join(r) for r in good) + "\n")

speakers = collections.Counter(r[1] for r in good)
NUM_SPEAKERS = len(speakers)
print(f"Kept: {len(good)} | Bad format: {len(bad)} | Missing wav: {missing} | Filtered duration (<{MIN_DURATION}s or >{MAX_DURATION}s): {too_long}")
print(f"Speakers ({NUM_SPEAKERS}):", speakers.most_common(15))

In [ ]:
# Download base checkpoint (LibriTTS-R medium, multi-speaker)
from huggingface_hub import hf_hub_download

BASE = hf_hub_download(
    BASE_REPO,
    BASE_CKPT_IN_REPO,
    repo_type="dataset",
    local_dir=f"{WORK}/base_ckpt",
)
print("Base checkpoint:", BASE)

In [ ]:
# Train on Kaggle GPU
# Note: --trainer.devices 1 is used so --data.length_bucketing true works reliably and avoids T4 OOM.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python3 -m piper.train fit \
  --data.voice_name {VOICE} \
  --data.csv_path {csv_path} \
  --data.audio_dir {audio_dir} \
  --data.phoneme_type vietnamese \
  --data.espeak_voice vi \
  --data.cache_dir {WORK}/cache \
  --data.config_path {OUT}/{VOICE}.onnx.json \
  --data.batch_size {BATCH_SIZE} \
  --data.num_workers 4 \
  --data.length_bucketing true \
  --model.sample_rate 22050 \
  --model.num_speakers {NUM_SPEAKERS} \
  --model.keep_last_k 3 \
  --model.mos_metric none \
  --model.warmstart_ckpt {BASE} \
  --trainer.devices 1 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.default_root_dir {OUT}

In [ ]:
# Resume training from last.ckpt (if continuing in the same session or from an uploaded checkpoint dataset)
# Set RESUME_CKPT to the path of last.ckpt and run this cell instead of the warmstart cell above.
import glob
ckpts = sorted(glob.glob(f"{OUT}/lightning_logs/*/checkpoints/last.ckpt"))
if ckpts:
    RESUME_CKPT = ckpts[-1]
    print("Resuming from:", RESUME_CKPT)
    !python3 -m piper.train fit \
      --data.voice_name {VOICE} \
      --data.csv_path {csv_path} \
      --data.audio_dir {audio_dir} \
      --data.phoneme_type vietnamese \
      --data.espeak_voice vi \
      --data.cache_dir {WORK}/cache \
      --data.config_path {OUT}/{VOICE}.onnx.json \
      --data.batch_size {BATCH_SIZE} \
      --data.num_workers 4 \
      --data.length_bucketing true \
      --model.sample_rate 22050 \
      --model.num_speakers {NUM_SPEAKERS} \
  --model.keep_last_k 3 \
      --model.mos_metric none \
      --ckpt_path {RESUME_CKPT} \
      --trainer.devices 1 \
      --trainer.max_epochs {MAX_EPOCHS} \
      --trainer.default_root_dir {OUT}

In [ ]:
# Export ONNX to /kaggle/working/piper_out
import glob
ckpt = sorted(glob.glob(f"{OUT}/lightning_logs/*/checkpoints/last.ckpt"))[-1]
print("Exporting checkpoint:", ckpt)
!python3 -m piper.train.export_onnx --checkpoint {ckpt} --output-file {OUT}/{VOICE}.onnx
!ls -lh {OUT}/{VOICE}.onnx {OUT}/{VOICE}.onnx.json

In [ ]:
# Test synthesis
!echo "Chương hai nghìn tám trăm mười. Xin chào, đây là giọng đọc thử nghiệm trên Kaggle." | python3 -m piper -m {OUT}/{VOICE}.onnx -s 0 -f /kaggle/working/test.wav
from IPython.display import Audio
Audio("/kaggle/working/test.wav")